# 🎙️ Deepfake Detection Pipeline

**Steps:**
1. Securely Clone Repo
2. Install Dependencies
3. Download/Link Data & Models
4. Run Audio Generation
5. Extract Features


## 1️⃣ Secure Setup & Clone

In [ ]:
# Mount Drive
from google.colab import drive
from getpass import getpass
import os
import shutil

drive.mount('/content/drive')

# --- CONFIGURATION ---
USERNAME = "charlesgrube-jpg"
REPO_NAME = "Data-Management-DDAA-KAN"
BRANCH = "feature/tts-vc-extraction"
# ---------------------

print("Enter your GitHub Personal Access Token (Permissions: repo):")
token = getpass()

# Secure URL construction
repo_url = f"https://{token}@github.com/{USERNAME}/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME):
    !git clone {repo_url} {REPO_NAME}
    %cd {REPO_NAME}
    !git checkout {BRANCH}
    print(f"✅ Cloned and checked out {BRANCH}!")
else:
    %cd {REPO_NAME}
    print("✅ Repo already exists.")

In [ ]:
# Install Deps
!apt-get install -y ffmpeg espeak-ng

# --- CRITICAL FIX: FORCE INSTALLATION BYPASS ---
# 1. Install Fairseq (ignoring broken metadata)
!pip install --no-deps git+https://github.com/facebookresearch/fairseq.git
# 2. Install RVC (ignoring dependencies to prevent re-triggering Fairseq check)
!pip install --no-deps git+https://github.com/MisileLab/rvc-python-butter.git
# ------------------------------------------------

# Use %pip to ensure installation in current kernel
# This installs all dependencies for RVC/Fairseq that we skipped above
%pip install -r requirements_colab.txt

## 2️⃣ Data & Model Setup
Choose **ONE** option below.

In [ ]:
import os
import shutil

# Ensure we are in the repo directory
REPO_NAME = "Data-Management-DDAA-KAN"
if os.path.exists(f"/content/{REPO_NAME}") and os.getcwd() != f"/content/{REPO_NAME}":
    %cd /content/{REPO_NAME}
    print(f"✅ Switched to /content/{REPO_NAME}")

# --- CHOOSE YOUR METHOD (Set True for ONLY ONE) ---
USE_OPTION_A_DIRECT_URL = False  # Link from Website
USE_OPTION_B_DRIVE_SYNC = False  # From Google Drive
USE_OPTION_C_MOZILLA_API = True  # Enter API Key/Token (Interactive)
# --------------------------------------------------

# OPTION A: DIRECT CLOUD DOWNLOAD (Fastest)
MOZILLA_URL = ""  # <-- Paste link here if using Option A

# OPTION B: SYNC FROM DRIVE
CUSTOM_SOURCE_PATH = ""  # e.g. "/content/drive/Othercomputers/My Laptop/en"

# ---------------------
MAX_FILES_TO_COPY = 50 # Set to 0 for FULL dataset (Only affects Option B)
# ---------------------

from tqdm.notebook import tqdm

TARGET_DIR = f"/content/{REPO_NAME}/mozilla_cv_data"

def smart_copy(src, dst):
    print(f"📂 Scanning source: {src} ...")
    all_files = []
    for root, dirs, files in os.walk(src):
        for file in files:
            all_files.append((os.path.join(root, file), os.path.relpath(os.path.join(root, file), src)))
    
    total_files = len(all_files)
    print(f"📊 Found {total_files} total files.")
    
    # Filtering
    tsvs = [f for f in all_files if f[0].endswith('.tsv')]
    if MAX_FILES_TO_COPY > 0 and total_files > MAX_FILES_TO_COPY:
        others = [f for f in all_files if not f[0].endswith('.tsv')]
        keep_others = others[:MAX_FILES_TO_COPY]
        files_to_copy = tsvs + keep_others
        print(f"✂️ LIMITING copy to {len(files_to_copy)} files.")
    else:
        files_to_copy = all_files

    if os.path.exists(dst):
        shutil.rmtree(dst)
    os.makedirs(dst, exist_ok=True)

    print("🚀 Starting copy...")
    for src_path, rel_path in tqdm(files_to_copy, unit="file"):
        dst_path = os.path.join(dst, rel_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(src_path, dst_path)
    print("✅ Copy Complete!")

def setup_data():
    os.makedirs(TARGET_DIR, exist_ok=True)

    # OPTION A: DIRECT URL
    if USE_OPTION_A_DIRECT_URL and MOZILLA_URL.startswith("http"):
        print("☁️ Starting Direct Cloud Download...")
        archive_path = "cv_corpus.tar.gz"
        !wget -O {archive_path} "{MOZILLA_URL}"
        print("📦 Extracting...")
        !tar -xzf {archive_path} -C {TARGET_DIR} --strip-components=1
        print("✅ Extraction Complete!")

    # OPTION C: MOZILLA API (INTERACTIVE)
    elif USE_OPTION_C_MOZILLA_API:
        print("🔑 Starting Interactive API Download...")
        if os.path.exists("utilities/download_mozilla_api.py"):
             !python utilities/download_mozilla_api.py
             
             # Check if download succeeded
             api_download_path = "mozilla_cv_data/common_voice_sample.tar.gz"
             if os.path.exists(api_download_path):
                  print("📦 Extracting API Download...")
                  !tar -xzf {api_download_path} -C {TARGET_DIR} --strip-components=1
                  print("✅ Extraction Complete!")
             else:
                  print("❌ API Download seemed to fail (File not found).")
        else:
             print("❌ ERROR: utilities/download_mozilla_api.py NOT FOUND. Make sure you are in the repo directory!")

    # OPTION B: DRIVE SYNC
    elif USE_OPTION_B_DRIVE_SYNC and CUSTOM_SOURCE_PATH:
        if os.path.isdir(CUSTOM_SOURCE_PATH):
             smart_copy(CUSTOM_SOURCE_PATH, TARGET_DIR)
        else:
             print("📦 Detected archive file. Extracting...")
             if CUSTOM_SOURCE_PATH.endswith(".tar.gz"):
                  !tar -xzf "{CUSTOM_SOURCE_PATH}" -C {TARGET_DIR}
             elif CUSTOM_SOURCE_PATH.endswith(".zip"):
                  !unzip -q "{CUSTOM_SOURCE_PATH}" -d {TARGET_DIR}
             print("✅ Extraction complete!")
        
    else:
        print("⚠️ No data source selected or configured!")

    # --- AUTO-FIX FOLDER STRUCTURE ---
    print("\n🔍 Verifying folder structure...")
    import glob
    found_clips = glob.glob(f"{TARGET_DIR}/**/clips", recursive=True)
    
    if found_clips:
        current_clips = found_clips[0]
        desired_clips = os.path.join(TARGET_DIR, "clips")
        
        if os.path.abspath(current_clips) != os.path.abspath(desired_clips):
            print(f"📦 Moving nested clips from {current_clips} to {desired_clips}")
            
            # Move contents of parent dir (likely contains TSVs too)
            parent_dir = os.path.dirname(current_clips)
            
            for item in os.listdir(parent_dir):
                src = os.path.join(parent_dir, item)
                dst = os.path.join(TARGET_DIR, item)
                if src != dst:
                    if os.path.isdir(src):
                        if os.path.exists(dst):
                            try:
                                !cp -r "{src}/." "{dst}/"
                                !rm -rf "{src}"
                            except:
                                pass
                        else:
                             shutil.move(src, dst)
                    else:
                        shutil.move(src, dst)
            print("✅ Fixed! Files have been flattened.")
    
    # Check Result
    if os.path.exists(os.path.join(TARGET_DIR, "clips")):
         print("✅ 'clips' folder ready.")
    else:
         print("❌ 'clips' folder NOT found in target.")

# RUN SETUP
setup_data()

# RUN RVC MODEL DOWNLOAD
print("\n📥 Downloading Standard RVC Research Models...")
if os.path.exists("utilities/download_rvc_models.py"):
    !python utilities/download_rvc_models.py
else:
    print("❌ ERROR: utilities/download_rvc_models.py NOT FOUND!")

## 3️⃣ Run Pipeline (Generate Audio)

In [ ]:
import yaml
import os

OUTPUT_PATH = "/content/drive/MyDrive/DDAA_Pipeline_Output"

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Configure for Colab
config['output']['base_dir'] = OUTPUT_PATH
config['source']['data_path'] = "mozilla_cv_data"

# --- LIMIT BY SIZE (New) ---
config['source']['max_size_mb'] = 100.0  # Limit input to 100MB of audio

# --- RVC CONFIGURATION ---
# Enable Voice Conversion
config['synthesis']['pick_strategy'] = "both" 
config['synthesis']['vc_models'] = ["DuaLipa", "TaylorSwift", "EdSheeran", "KanyeWest"]
config['synthesis']['vc_device'] = "cuda:0" # Set to "cpu" if GPU quota exceeded

config['codec_compression']['enabled'] = True

with open("config.yaml", "w") as f:
    yaml.dump(config, f)

print("✅ Config updated: Enabled RVC & Size Limit (100MB)!")
!python run_pipeline.py

## 4️⃣ Extract Features

In [ ]:
import os
import glob

# 1. Find the latest output folder in Drive
drive_output_pattern = "/content/drive/MyDrive/DDAA_Pipeline_Output_*"
list_of_folders = glob.glob(drive_output_pattern)

if not list_of_folders:
    print("❌ No output folder found! Did the pipeline run successfully?")
else:
    latest_folder = max(list_of_folders, key=os.path.getctime)
    print(f"✅ Found latest dataset: {latest_folder}")
    
    # 2. Define output feature folder inside the dataset folder
    output_features = os.path.join(latest_folder, "features_cqt")

    print(f"🚀 Extracting CQT features to: {output_features}")
    
    # 3. Run Extraction
    !python -m pipeline.features.extract_features \
        --type cqt \
        --input "{latest_folder}" \
        --output "{output_features}"